## Carga y preparación inicial del dataset

Se realiza la carga del conjunto de datos climático. Se importa el archivo CSV original y posteriormente se generan copias en diferentes etapas del pipeline (`interim` y `processed`) con el objetivo de mantener una estructura organizada del proyecto y preservar versiones del dataset a medida que avanza el procesamiento.

Posteriormente, la columna `DATE` es convertida al tipo `datetime`, lo que permite trabajar correctamente con operaciones temporales como filtrados, agregaciones y análisis de series de tiempo.

Luego se aplica un filtro temporal para conservar únicamente los registros comprendidos entre los años 2020 y 2025. Esta decisión permite enfocar el análisis en información reciente y consistente, reduciendo posibles sesgos generados por datos históricos demasiado antiguos.

Finalmente, se reinicia el índice del DataFrame luego del filtrado para mantener una estructura ordenada y consecutiva de los registros.

Como resultado de esta preparación inicial, el dataset queda conformado por **2192 registros y 144 variables**, mostrando información meteorológica diaria correspondiente a la estación analizada.

In [4]:
import pandas as pd
import numpy as np

# Carga
df = pd.read_csv("../../data/raw/USW00014607.csv", low_memory=False)

# Guardado de datos limpios
df.to_csv("../../data/interim/weather_cleaned.csv", index=False)

# Guardado de datos procesados
df.to_csv("../../data/processed/weather_processed.csv", index=False)

# Parseo de fecha
df["DATE"] = pd.to_datetime(df["DATE"])

# Filtro temporal
df = df[(df["DATE"] >= "2020-01-01") & (df["DATE"] <= "2025-12-31")]

df.reset_index(drop=True, inplace=True)

print(df.shape)
df.head()

(2192, 144)


,STATION,DATE,LATITUDE,LONGITUDE,ELEVATION,NAME,PRCP,PRCP_ATTRIBUTES,SNOW,SNOW_ATTRIBUTES,...,WT19,WT19_ATTRIBUTES,WT21,WT21_ATTRIBUTES,WT22,WT22_ATTRIBUTES,WV03,WV03_ATTRIBUTES,WV20,WV20_ATTRIBUTES
0,USW00014607,2020-01-01,46.87049,-68.01723,188.6,"CARIBOU WEATHER FORECAST OFFICE, ME US",10.0,",,W,2400",13.0,",,W,2400",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,USW00014607,2020-01-02,46.87049,-68.01723,188.6,"CARIBOU WEATHER FORECAST OFFICE, ME US",0.0,"T,,W,2400",0.0,"T,,W,2400",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,USW00014607,2020-01-03,46.87049,-68.01723,188.6,"CARIBOU WEATHER FORECAST OFFICE, ME US",0.0,"T,,W,2400",3.0,",,W,2400",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,USW00014607,2020-01-04,46.87049,-68.01723,188.6,"CARIBOU WEATHER FORECAST OFFICE, ME US",0.0,"T,,W,2400",0.0,"T,,W,2400",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,USW00014607,2020-01-05,46.87049,-68.01723,188.6,"CARIBOU WEATHER FORECAST OFFICE, ME US",13.0,",,W,2400",18.0,",,W,2400",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Construcción de la variable objetivo y reducción de ruido

Se construye la variable objetivo `Rain`, la cual representa si ocurrió precipitación en un determinado día. Para ello, se utiliza la columna `PRCP`, asignando el valor `1` cuando la precipitación es mayor a cero y `0` en caso contrario. De esta manera, el problema queda formulado como una tarea de clasificación binaria.

Posteriormente, la variable `PRCP` es eliminada del dataset para evitar *data leakage*. Esto se debe a que dicha columna contiene directamente la información utilizada para construir el target, por lo que mantenerla dentro de las variables predictoras permitiría al modelo “anticipar” la respuesta de forma artificial, generando métricas irreales y un sobreajuste significativo.

A continuación, se eliminan las columnas que contienen la palabra `ATTRIBUTES`, ya que corresponden principalmente a metadatos técnicos asociados a la calidad o procedencia de las mediciones meteorológicas. Estas variables no aportan valor predictivo relevante para el objetivo planteado y aumentan innecesariamente la dimensionalidad del dataset.

Finalmente, también se descartan las columnas cuyos nombres comienzan con `WT` y `WV`. Estas variables representan flags o indicadores climáticos altamente dispersos y con una gran cantidad de valores faltantes, lo que introduce ruido y complejidad adicional en el entrenamiento. La decisión busca simplificar el conjunto de datos y conservar únicamente variables con mayor capacidad explicativa y consistencia estadística.

In [ ]:
# Target: lluvia (1 si llueve, 0 si no)
df["Rain"] = (df["PRCP"] > 0).astype(int)

# Eliminamos PRCP si queremos evitar leakage directo
# (esto lo vamos a comparar después, pero por ahora lo sacamos)
df = df.drop(columns=["PRCP"])

In [6]:
# Eliminar columnas tipo attributes
attr_cols = [col for col in df.columns if "ATTRIBUTES" in col]
df = df.drop(columns=attr_cols)

# Eliminar columnas de flags muy ruidosas (evaluamos luego si sumar alguna)
wt_cols = [col for col in df.columns if col.startswith("WT")]
wv_cols = [col for col in df.columns if col.startswith("WV")]

df = df.drop(columns=wt_cols + wv_cols)

## Tratamiento de valores faltantes y variables no informativas

Se analiza la proporción de valores faltantes presentes en cada variable del dataset. Para ello, se calcula el porcentaje de `NaN` por columna utilizando la media de valores nulos.

A partir de este análisis, se eliminan aquellas variables que poseen más del 50% de datos faltantes. Este criterio permite reducir el impacto de información incompleta que podría afectar negativamente la calidad del entrenamiento, además de evitar imputaciones excesivas que introduzcan ruido o sesgos artificiales.

Como resultado de este proceso, se eliminaron **29 columnas** con un nivel elevado de valores ausentes.

Posteriormente, se identifican y eliminan las columnas constantes, es decir, aquellas variables que presentan un único valor en todos los registros. Este tipo de variables no aporta capacidad predictiva al modelo, ya que no contienen variabilidad ni información útil para diferenciar observaciones.

En total, se removieron **5 columnas constantes**, contribuyendo a simplificar el dataset y mejorar la eficiencia del proceso de modelado.

In [7]:
# % de NaNs
missing_ratio = df.isna().mean()

# Eliminar columnas con muchos NaNs (>50%)
cols_to_drop = missing_ratio[missing_ratio > 0.5].index
df = df.drop(columns=cols_to_drop)

print(f"Columnas eliminadas por NaN: {len(cols_to_drop)}")

Columnas eliminadas por NaN: 29


In [8]:
# Columnas sin variación
constant_cols = [col for col in df.columns if df[col].nunique() <= 1]
df = df.drop(columns=constant_cols)

print(f"Columnas constantes eliminadas: {len(constant_cols)}") 

Columnas constantes eliminadas: 5


## Ingeniería de variables temporales

En esta etapa se realiza un proceso de *feature engineering* orientado a extraer información temporal relevante a partir de la columna `DATE`.

Se generan nuevas variables derivadas como el año (`year`), mes (`month`), día (`day`) y día de la semana (`dayofweek`). Estas características permiten que el modelo capture patrones temporales y comportamientos estacionales asociados a las condiciones climáticas.

Además, se incorporan las variables `month_sin` y `month_cos`, construidas mediante transformaciones trigonométricas sobre el mes del año. Esta técnica permite representar la naturaleza cíclica de las estaciones, evitando que el modelo interprete diciembre y enero como períodos alejados entre sí cuando, en realidad, son meses consecutivos dentro del ciclo anual.

Posteriormente, la columna original `DATE` es eliminada del dataset, ya que los componentes temporales relevantes ya fueron extraídos y transformados en variables numéricas más adecuadas para el entrenamiento de modelos de Machine Learning.

Finalmente, el dataset procesado es almacenado nuevamente en la carpeta `interim`, preservando una versión limpia y enriquecida para las siguientes etapas del pipeline analítico.

In [ ]:
df["year"] = df["DATE"].dt.year
df["month"] = df["DATE"].dt.month
df["day"] = df["DATE"].dt.day
df["dayofweek"] = df["DATE"].dt.dayofweek

# Opcional: estacionalidad (mejor que mes crudo)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

# Eliminamos DATE (no usable directo)
df = df.drop(columns=["DATE"])

In [13]:
df.to_csv("../../data/interim/weather_cleaned.csv", index=False)

## Separación de variables y partición del dataset

En esta etapa se separan las variables predictoras (`X`) de la variable objetivo (`y`). La columna `Rain` es utilizada como target del modelo, mientras que el resto de las variables conforman el conjunto de características utilizadas para realizar las predicciones.

Luego de todo el proceso de limpieza y transformación, el dataset final queda compuesto por **2192 registros y 22 variables predictoras**, reduciendo significativamente la dimensionalidad inicial y conservando únicamente información relevante para el problema.

Posteriormente, se realiza la división del dataset en conjuntos de entrenamiento y prueba mediante la función `train_test_split`. Se utiliza un 80% de los datos para entrenamiento y un 20% para evaluación.

Además, se aplica el parámetro `stratify=y`, lo que garantiza que ambas particiones mantengan una distribución similar de la variable objetivo. Esto resulta especialmente importante en problemas de clasificación binaria, ya que evita desbalances entre entrenamiento y prueba que puedan sesgar la evaluación del modelo.

También se fija una semilla aleatoria (`random_state=42`) para asegurar la reproducibilidad de los resultados.

### Resumen del dataset procesado

- Período analizado: **2020 - 2025**
- Registros totales: **2192**
- Variables originales: **~144**
- Variables finales: **22**

Durante el proceso de preparación se realizaron las siguientes tareas:

- limpieza de columnas irrelevantes
- tratamiento de valores faltantes
- eliminación de variables constantes
- reducción de ruido y dimensionalidad
- ingeniería de variables temporales

Finalmente, el dataset procesado es almacenado en la carpeta `processed`, dejando disponible una versión lista para el modelado y entrenamiento de algoritmos de Machine Learning pero no sin antes pasar por una ingenieria de atributos.

In [14]:
X = df.drop(columns=["Rain"])
y = df["Rain"]

print(X.shape)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

(2192, 22)


In [15]:
df.to_csv("../../data/processed/weather_processed.csv", index=False)